# Join de pedidos, feriados e eventos de e-commerce — Last Mile SP ↔ SP

Este notebook parte do pressuposto de que a ETL anterior **já filtrou apenas pedidos em que vendedor e cliente estão no estado de São Paulo (SP ↔ SP)**.

O objetivo é enriquecer os pedidos com:

- feriados oficiais;
- distância até o feriado anterior e o próximo;
- janelas de 7 e 14 dias em torno de feriados;
- eventos comerciais relevantes para o e-commerce brasileiro;
- distância até o evento comercial anterior e o próximo;
- janelas de 7 e 14 dias em torno desses eventos;
- análise exploratória da taxa de atraso próxima aos eventos.

Como o escopo é **Last Mile**, todas as features temporais são relacionadas a `order_delivered_carrier_date`, utilizada no projeto como referência de entrada do pedido na etapa da transportadora.

> **Importante:** a taxa histórica de atraso por evento é calculada apenas para análise exploratória. Ela não é adicionada como feature ao dataset final, evitando target leakage.


In [ ]:
%pip install pyspark

## 1. Inicialização do PySpark

Inicializa a sessão Spark com suporte ao acesso aos buckets S3 utilizados no projeto.


In [ ]:
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import date

conf = SparkConf()
conf.set(
    "spark.jars.packages",
    "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.11.901"
)
conf.set(
    "spark.hadoop.fs.s3a.aws.credentials.provider",
    "com.amazonaws.auth.InstanceProfileCredentialsProvider"
)

spark = SparkSession.builder.config(conf=conf).getOrCreate()


## 2. Leitura das tabelas

A tabela de pedidos já deve estar filtrada pela ETL para **SP ↔ SP**.

A tabela `tb_dates` contém os feriados utilizados no projeto.


In [ ]:
tb_orders = spark.read.csv(
    "s3a://last-mile-optimization-trusted-sofhia/dataset-orders/final_table_orders_numeric.csv/",
    header=True,
    inferSchema=True
)

tb_dates = spark.read.csv(
    "s3a://last-mile-optimization-trusted-sofhia/dataset-holidays/dates_2016_2018/",
    header=True,
    inferSchema=True
)

print("Quantidade de pedidos:", tb_orders.count())
print("Quantidade de registros de feriados:", tb_dates.count())

tb_dates.select("date", "localName").show(50, truncate=False)


## 3. Preparação das datas

Os CSVs de feriados enviados possuem datas no formato `yyyy-MM-dd`, mas o código também aceita `dd/MM/yyyy`.

Para os pedidos, criamos `carrier_date_only` a partir de `order_delivered_carrier_date`. Essa será a data de referência das features porque o estudo está focado no **Last Mile**.


In [ ]:
# Aceita tanto yyyy-MM-dd quanto dd/MM/yyyy
tb_dates_prep = tb_dates.withColumn(
    "data_feriado",
    F.coalesce(
        F.to_date(F.col("date"), "yyyy-MM-dd"),
        F.to_date(F.col("date"), "dd/MM/yyyy")
    )
)

tb_orders_prep = tb_orders.withColumn(
    "carrier_date_only",
    F.to_date(F.col("order_delivered_carrier_date"))
)

tb_dates_prep.select("date", "localName", "data_feriado").show(50, truncate=False)


## 4. Eventos comerciais relevantes para o e-commerce brasileiro

Feriados oficiais e eventos comerciais são tratados separadamente.

Além de datas como Natal, incluímos eventos que não são feriados oficiais, mas podem gerar aumento de demanda no e-commerce:

- Dia das Mães;
- Dia dos Pais;
- Dia das Crianças;
- Black Friday;
- Cyber Monday;
- Natal.

Não atribuímos pesos manuais como “Black Friday = 3”. O modelo receberá características temporais objetivas e poderá aprender sozinho a relevância de cada período.


In [ ]:
ecommerce_events = [
    # 2016
    (date(2016, 5, 8),  "Dia das Maes"),
    (date(2016, 8, 14), "Dia dos Pais"),
    (date(2016, 10, 12), "Dia das Criancas"),
    (date(2016, 11, 25), "Black Friday"),
    (date(2016, 11, 28), "Cyber Monday"),
    (date(2016, 12, 25), "Natal"),

    # 2017
    (date(2017, 5, 14), "Dia das Maes"),
    (date(2017, 8, 13), "Dia dos Pais"),
    (date(2017, 10, 12), "Dia das Criancas"),
    (date(2017, 11, 24), "Black Friday"),
    (date(2017, 11, 27), "Cyber Monday"),
    (date(2017, 12, 25), "Natal"),

    # 2018
    (date(2018, 5, 13), "Dia das Maes"),
    (date(2018, 8, 12), "Dia dos Pais"),
    (date(2018, 10, 12), "Dia das Criancas"),
    (date(2018, 11, 23), "Black Friday"),
    (date(2018, 11, 26), "Cyber Monday"),
    (date(2018, 12, 25), "Natal")
]

tb_ecommerce_events = spark.createDataFrame(
    ecommerce_events,
    ["event_date", "event_name"]
)

tb_ecommerce_events.orderBy("event_date").show(30, truncate=False)


## 5. Criação do calendário contínuo

Assim como no notebook original, criamos uma dimensão diária contínua. Isso evita fazer vários joins pesados diretamente entre pedidos, feriados e eventos.

O intervalo de 2015 a 2019 garante margem suficiente em torno dos dados de 2016–2018.


In [ ]:
start_date = F.to_date(F.lit("2015-01-01"))
end_date = F.to_date(F.lit("2019-12-31"))

calendar_df = spark.range(1).select(
    F.explode(
        F.sequence(start_date, end_date, F.expr("interval 1 day"))
    ).alias("calendar_date")
)

calendar_df.show(5)


## 6. Features de feriados oficiais

Para cada dia do calendário são calculados:

- se o próprio dia é feriado;
- dias desde o último feriado;
- dias até o próximo feriado;
- presença de feriado nos próximos 7 e 14 dias;
- ocorrência de feriado nos últimos 7 e 14 dias.


In [ ]:
calendar_with_holidays = calendar_df.join(
    tb_dates_prep.select("data_feriado").dropDuplicates(),
    calendar_df.calendar_date == tb_dates_prep.data_feriado,
    "left"
).withColumn(
    "is_holiday_today",
    F.when(F.col("data_feriado").isNotNull(), 1).otherwise(0)
).select(
    "calendar_date",
    "is_holiday_today"
)

window_past = Window.orderBy("calendar_date").rowsBetween(
    Window.unboundedPreceding, 0
)

window_future = Window.orderBy("calendar_date").rowsBetween(
    0, Window.unboundedFollowing
)

calendar_holiday_features = calendar_with_holidays.withColumn(
    "last_holiday_date",
    F.max(
        F.when(F.col("is_holiday_today") == 1, F.col("calendar_date"))
    ).over(window_past)
).withColumn(
    "next_holiday_date",
    F.min(
        F.when(F.col("is_holiday_today") == 1, F.col("calendar_date"))
    ).over(window_future)
)

calendar_holiday_features = calendar_holiday_features.withColumn(
    "days_since_last_holiday",
    F.coalesce(
        F.datediff(F.col("calendar_date"), F.col("last_holiday_date")),
        F.lit(999)
    )
).withColumn(
    "days_until_next_holiday",
    F.coalesce(
        F.datediff(F.col("next_holiday_date"), F.col("calendar_date")),
        F.lit(999)
    )
).withColumn(
    "is_holiday_in_7_days",
    F.when(F.col("days_until_next_holiday") <= 7, 1).otherwise(0)
).withColumn(
    "is_holiday_in_14_days",
    F.when(F.col("days_until_next_holiday") <= 14, 1).otherwise(0)
).withColumn(
    "had_holiday_7_days_ago",
    F.when(F.col("days_since_last_holiday") <= 7, 1).otherwise(0)
).withColumn(
    "had_holiday_14_days_ago",
    F.when(F.col("days_since_last_holiday") <= 14, 1).otherwise(0)
)

calendar_holiday_features.show(10)


## 7. Features dos eventos de e-commerce

Aplicamos a mesma lógica aos eventos comerciais.

As novas features serão:

- `ecommerce_event_at_carrier`;
- `previous_ecommerce_event`;
- `next_ecommerce_event`;
- `days_since_last_ecommerce_event`;
- `days_until_next_ecommerce_event`;
- `is_ecommerce_event_in_7_days`;
- `is_ecommerce_event_in_14_days`;
- `had_ecommerce_event_7_days_ago`;
- `had_ecommerce_event_14_days_ago`.

Os nomes do evento anterior/próximo são mantidos para análise. Antes de treinar modelos que exigem somente números, essas colunas podem ser removidas ou codificadas.


In [ ]:
calendar_with_events = calendar_df.join(
    tb_ecommerce_events,
    calendar_df.calendar_date == tb_ecommerce_events.event_date,
    "left"
).withColumn(
    "is_ecommerce_event_today",
    F.when(F.col("event_date").isNotNull(), 1).otherwise(0)
)

calendar_event_features = calendar_with_events.withColumn(
    "last_ecommerce_event_date",
    F.max(
        F.when(
            F.col("is_ecommerce_event_today") == 1,
            F.col("calendar_date")
        )
    ).over(window_past)
).withColumn(
    "next_ecommerce_event_date",
    F.min(
        F.when(
            F.col("is_ecommerce_event_today") == 1,
            F.col("calendar_date")
        )
    ).over(window_future)
)

# Mapeamento data -> nome para recuperar os nomes do evento anterior/próximo
event_lookup_previous = tb_ecommerce_events.select(
    F.col("event_date").alias("previous_event_date_lookup"),
    F.col("event_name").alias("previous_ecommerce_event")
)

event_lookup_next = tb_ecommerce_events.select(
    F.col("event_date").alias("next_event_date_lookup"),
    F.col("event_name").alias("next_ecommerce_event")
)

calendar_event_features = calendar_event_features.join(
    event_lookup_previous,
    calendar_event_features.last_ecommerce_event_date ==
    event_lookup_previous.previous_event_date_lookup,
    "left"
).join(
    event_lookup_next,
    calendar_event_features.next_ecommerce_event_date ==
    event_lookup_next.next_event_date_lookup,
    "left"
)

calendar_event_features = calendar_event_features.withColumn(
    "days_since_last_ecommerce_event",
    F.coalesce(
        F.datediff(
            F.col("calendar_date"),
            F.col("last_ecommerce_event_date")
        ),
        F.lit(999)
    )
).withColumn(
    "days_until_next_ecommerce_event",
    F.coalesce(
        F.datediff(
            F.col("next_ecommerce_event_date"),
            F.col("calendar_date")
        ),
        F.lit(999)
    )
).withColumn(
    "is_ecommerce_event_in_7_days",
    F.when(F.col("days_until_next_ecommerce_event") <= 7, 1).otherwise(0)
).withColumn(
    "is_ecommerce_event_in_14_days",
    F.when(F.col("days_until_next_ecommerce_event") <= 14, 1).otherwise(0)
).withColumn(
    "had_ecommerce_event_7_days_ago",
    F.when(F.col("days_since_last_ecommerce_event") <= 7, 1).otherwise(0)
).withColumn(
    "had_ecommerce_event_14_days_ago",
    F.when(F.col("days_since_last_ecommerce_event") <= 14, 1).otherwise(0)
)

calendar_event_features.select(
    "calendar_date",
    "is_ecommerce_event_today",
    "previous_ecommerce_event",
    "next_ecommerce_event",
    "days_since_last_ecommerce_event",
    "days_until_next_ecommerce_event",
    "is_ecommerce_event_in_7_days",
    "is_ecommerce_event_in_14_days",
    "had_ecommerce_event_7_days_ago",
    "had_ecommerce_event_14_days_ago"
).filter(
    (F.col("days_since_last_ecommerce_event") <= 3) |
    (F.col("days_until_next_ecommerce_event") <= 3)
).show(50, truncate=False)


## 8. União das features em uma única dimensão de calendário

Aqui juntamos as features de feriados e de eventos comerciais. Depois será necessário apenas um join dessa dimensão com os pedidos.


In [ ]:
calendar_final_features = calendar_holiday_features.select(
    "calendar_date",
    "is_holiday_today",
    "days_since_last_holiday",
    "days_until_next_holiday",
    "is_holiday_in_7_days",
    "is_holiday_in_14_days",
    "had_holiday_7_days_ago",
    "had_holiday_14_days_ago"
).join(
    calendar_event_features.select(
        "calendar_date",
        "is_ecommerce_event_today",
        "previous_ecommerce_event",
        "next_ecommerce_event",
        "days_since_last_ecommerce_event",
        "days_until_next_ecommerce_event",
        "is_ecommerce_event_in_7_days",
        "is_ecommerce_event_in_14_days",
        "had_ecommerce_event_7_days_ago",
        "had_ecommerce_event_14_days_ago"
    ),
    "calendar_date",
    "left"
).withColumnRenamed(
    "calendar_date",
    "match_date"
)

calendar_final_features.show(10, truncate=False)


## 9. Join com os pedidos

O join é realizado entre `carrier_date_only` e o calendário enriquecido.

Dessa forma, as features representam o contexto temporal existente quando o pedido entra na etapa da transportadora.


In [ ]:
df_final = tb_orders_prep.join(
    calendar_final_features,
    tb_orders_prep.carrier_date_only == calendar_final_features.match_date,
    "left"
).withColumn(
    "holiday_at_carrier",
    F.coalesce(F.col("is_holiday_today"), F.lit(0))
).withColumn(
    "ecommerce_event_at_carrier",
    F.coalesce(F.col("is_ecommerce_event_today"), F.lit(0))
)

df_final = df_final.select(
    # Datas originais
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "interval_code_delivered_carrier",
    "order_delivered_customer_date",
    "interval_code_delivered_customer",
    "order_estimated_delivery_date",
    "shipping_limit_date",

    # Features de feriados
    "holiday_at_carrier",
    "days_since_last_holiday",
    "days_until_next_holiday",
    "is_holiday_in_7_days",
    "is_holiday_in_14_days",
    "had_holiday_7_days_ago",
    "had_holiday_14_days_ago",

    # Features de eventos de e-commerce
    "ecommerce_event_at_carrier",
    "previous_ecommerce_event",
    "next_ecommerce_event",
    "days_since_last_ecommerce_event",
    "days_until_next_ecommerce_event",
    "is_ecommerce_event_in_7_days",
    "is_ecommerce_event_in_14_days",
    "had_ecommerce_event_7_days_ago",
    "had_ecommerce_event_14_days_ago",

    # Demais features já existentes
    "purchase_hour",
    "purchase_day_of_week",
    "is_weekend",
    "delivered_on_time",
    "delivery_time_days",
    "price",
    "freight_value",
    "product_weight_g",
    "volume_cm3",
    "distance_km",
    "category_name",
    "customer_city",
    "seller_city",
    "same_city",
    "review_score"
).dropna(
    subset=["product_weight_g", "volume_cm3"]
)

df_final.show(10, truncate=False)


## 10. Validação das novas features

As células abaixo permitem conferir quantos pedidos entraram no last mile exatamente em feriados ou eventos comerciais.


In [ ]:
print("Pedidos por holiday_at_carrier:")
df_final.groupBy("holiday_at_carrier").count().show()

print("Pedidos por ecommerce_event_at_carrier:")
df_final.groupBy("ecommerce_event_at_carrier").count().show()


In [ ]:
# Pedidos cuja entrada no last mile ocorreu exatamente em um evento comercial
df_final.filter(
    F.col("ecommerce_event_at_carrier") == 1
).select(
    "order_delivered_carrier_date",
    "previous_ecommerce_event",
    "next_ecommerce_event",
    "delivered_on_time"
).show(50, truncate=False)


## 11. Análise exploratória: atraso próximo aos eventos

Esta etapa responde à pergunta:

**Pedidos que entram no last mile próximos de Black Friday, Natal, Dia das Mães etc. apresentam maior taxa de atraso?**

Usaremos inicialmente uma janela de:

- até **14 dias antes** do evento;
- até **7 dias depois** do evento.

A variável `delivered_on_time` é usada apenas para medir o resultado histórico.

> **Esta taxa não será incorporada como feature no dataset final.** Fazer isso usando toda a base poderia introduzir target leakage.


In [ ]:
# Para esta análise exploratória, cruzamos somente as datas necessárias.
orders_events_analysis = tb_orders_prep.select(
    "order_delivered_carrier_date",
    "carrier_date_only",
    "delivered_on_time"
).crossJoin(
    F.broadcast(tb_ecommerce_events)
).withColumn(
    "days_from_event",
    F.datediff(
        F.col("carrier_date_only"),
        F.col("event_date")
    )
)

orders_near_events = orders_events_analysis.filter(
    (F.col("days_from_event") >= -14) &
    (F.col("days_from_event") <= 7)
)

orders_near_events.select(
    "carrier_date_only",
    "event_date",
    "event_name",
    "days_from_event",
    "delivered_on_time"
).show(20, truncate=False)


### Taxa de atraso por evento

Considerando a codificação utilizada no projeto:

- `delivered_on_time = 1` → entrega no prazo;
- `delivered_on_time = 0` → entrega atrasada.

A tabela abaixo calcula a quantidade de pedidos e a taxa de atraso observada em torno de cada evento.


In [ ]:
event_analysis = orders_near_events.groupBy(
    "event_name"
).agg(
    F.count("*").alias("total_orders"),
    F.sum(
        F.when(F.col("delivered_on_time") == 0, 1).otherwise(0)
    ).alias("delayed_orders")
).withColumn(
    "delay_rate",
    F.round(
        F.col("delayed_orders") / F.col("total_orders"),
        4
    )
).withColumn(
    "delay_rate_percent",
    F.round(F.col("delay_rate") * 100, 2)
)

event_analysis.orderBy(
    F.desc("delay_rate")
).show(truncate=False)


### Separação entre período anterior e posterior ao evento

Para Last Mile, essa comparação pode ser útil porque a pressão logística pode se manifestar de maneira diferente antes e depois de uma data comercial.

- `PRE_EVENT` → pedido entrou no last mile entre 14 e 1 dia antes;
- `EVENT_DAY` → entrou exatamente na data do evento;
- `POST_EVENT` → entrou entre 1 e 7 dias depois.


In [ ]:
orders_near_events_period = orders_near_events.withColumn(
    "event_period",
    F.when(F.col("days_from_event") < 0, "PRE_EVENT")
     .when(F.col("days_from_event") == 0, "EVENT_DAY")
     .otherwise("POST_EVENT")
)

event_period_analysis = orders_near_events_period.groupBy(
    "event_name",
    "event_period"
).agg(
    F.count("*").alias("total_orders"),
    F.sum(
        F.when(F.col("delivered_on_time") == 0, 1).otherwise(0)
    ).alias("delayed_orders")
).withColumn(
    "delay_rate_percent",
    F.round(
        (F.col("delayed_orders") / F.col("total_orders")) * 100,
        2
    )
)

event_period_analysis.orderBy(
    "event_name",
    "event_period"
).show(100, truncate=False)


## 12. Comparação com a taxa geral de atraso

A taxa geral serve como referência para interpretar os eventos.

Por exemplo, se a taxa geral de atraso for 8% e a janela da Black Friday apresentar 15%, existe um sinal descritivo de maior concentração de atrasos naquele período. Isso **não prova causalidade**, mas pode justificar investigar a variável no modelo.


In [ ]:
overall_delay = df_final.agg(
    F.count("*").alias("total_orders"),
    F.sum(
        F.when(F.col("delivered_on_time") == 0, 1).otherwise(0)
    ).alias("delayed_orders")
).withColumn(
    "overall_delay_rate_percent",
    F.round(
        (F.col("delayed_orders") / F.col("total_orders")) * 100,
        2
    )
)

overall_delay.show(truncate=False)


## 13. Verificação de valores nulos

Confere se alguma das features ficou sem valor após os joins.


In [ ]:
null_counts = [
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df_final.columns
]

df_final.select(null_counts).show(truncate=False)


## 14. Dataset numérico para Machine Learning

`previous_ecommerce_event` e `next_ecommerce_event` são colunas categóricas em texto.

Para um treinamento que utilizará somente as variáveis numéricas, podemos removê-las. As demais features temporais dos eventos continuam disponíveis para o XGBoost.

Se posteriormente houver interesse em avaliar o nome do evento como variável, ele deve ser codificado adequadamente dentro do pipeline de treinamento.


In [ ]:
df_final_ml = df_final.drop(
    "previous_ecommerce_event",
    "next_ecommerce_event"
)

print("Colunas disponíveis para a próxima etapa:")
for c in df_final_ml.columns:
    print(c)


## 15. Exportação para a camada Client

O arquivo final contém as features originais de feriados e as novas features relacionadas aos eventos comerciais.

A análise de `delay_rate` não é exportada como feature.


In [ ]:
df_final_ml.coalesce(1) \
    .write \
    .option("header", "true") \
    .mode("overwrite") \
    .csv(
        "s3a://last-mile-optimization-client-sofhia/table_orders_holidays_ecommerce_events.csv"
    )

print("Dataset exportado com sucesso.")


## 16. Encerramento

Ao finalizar todas as operações, encerramos a sessão Spark.


In [ ]:
spark.stop()
